<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/notebooks/02_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIND-small: canonical data preprocessing

Run from top to bottom to produce the shared CSV handoff. EDA lives in `notebooks/eda/`; modelling starts from the exported CSVs. The helpers in `src/preprocessing.py` contain the common loading, parsing and integrity checks. See README.md for local/Colab setup.

## 1. Configure paths

Raw data is unchanged. Set `MIND_RAW_DIR` for a shared Drive folder and `BT4222_PROJECT_ROOT` if the notebook is outside the clone.

In [1]:
from pathlib import Path
import os
import subprocess

repo_root = Path('/content/bt4222')

if not (repo_root / 'src/preprocessing.py').is_file():
    subprocess.run(
        [
            'git', 'clone',
            'https://github.com/JariusF10-8/bt4222.git',
            str(repo_root),
        ],
        check=True,
    )

os.environ['BT4222_PROJECT_ROOT'] = str(repo_root)
os.environ['MIND_RAW_DIR'] = str(repo_root / 'data/raw')

In [2]:
import importlib
from pathlib import Path
import os
import sys
from IPython.display import display

# import src
# importlib.reload(src.preprocessing)

# In Colab, clone/open the repository first; set BT4222_PROJECT_ROOT if needed.
# For shared Drive data, set MIND_RAW_DIR to the folder containing MINDsmall_train.
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")

configured_root = os.environ.get("BT4222_PROJECT_ROOT")
if configured_root:
    PROJECT_ROOT = Path(configured_root).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / "src/data_loading.py").is_file()), None)
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src/data_loading.py").is_file():
    raise FileNotFoundError("Open the cloned repo, or set BT4222_PROJECT_ROOT to its path.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = Path(os.environ.get("MIND_RAW_DIR", str(PROJECT_ROOT / "data/raw"))).expanduser()


Mounted at /content/drive


## 2. Choose a new output version

Configure a new output version in Google Drive. REQUIRE_TEST=True requires the complete train/validation/test handoff. Existing output folders are preserved; choose an unused version for each reproduction build.

In [3]:
STORAGE_ROOT = Path(
    '/content/drive/MyDrive/BT4222 Group Project'
)

# Separate reproduction check; preserves the agreed processed/v2 bundle, original preprocessed outputs in v2.
PROCESSED_VERSION = 'v2_colab_check'
OUTPUT_DIR = STORAGE_ROOT / 'data/processed' / PROCESSED_VERSION

REQUIRE_TEST = True

print('Raw data:', DATA_ROOT)
print('Output:', OUTPUT_DIR)

Raw data: /content/bt4222/data/raw
Output: /content/drive/MyDrive/BT4222 Group Project/data/processed/v2_colab_check


## 3. Prepare, validate and export

- Read literal quotes correctly; reject invalid IDs, timestamps, candidate tokens and malformed entity JSON.
- Keep the supplied prior history and every candidate in original order. No negative sampling here: the export stays complete (see section 4).
- Train: November 9–12, 2019; validation: November 13–14; test: all dev events, verified strictly later. Split whole events before expanding candidates.
- Retain metadata for candidates and history articles from the appropriate source only. Preserve original text, flag missingness, parse entities and calculate whitespace word lengths. Missing metadata gets explicit placeholders.
- Verify chronology, event uniqueness, candidate reconstruction and metadata joins. Export separate event/candidate/news CSVs with JSON list columns and portable string event keys.

The manifest records source checksums, code checksum, versions and completeness. Files are published only after all partitions pass. Timestamps have no assumed timezone.

In [4]:
from src.preprocessing import build_dataset

split_summary = build_dataset(
    DATA_ROOT,
    OUTPUT_DIR,
    require_test=REQUIRE_TEST,
    overwrite=False,
)

display(split_summary)

Preparing train: 95,071 events
Preparing validation: 61,894 events
Preparing test: 73,152 events


,partition,earliest_time,latest_time,events,users,candidate_exposures,clicked_candidates,nonclicked_candidates,events_with_empty_history,referenced_articles,articles_missing_metadata,candidate_exposures_missing_metadata
0,train,2019-11-09 00:00:19,2019-11-12 23:59:25,95071,40148,3389342,141558,3247784,1735,43370,0,0
1,validation,2019-11-13 00:01:00,2019-11-14 23:59:13,61894,31664,2454102,94786,2359316,1503,38421,0,0
2,test,2019-11-15 00:00:01,2019-11-15 23:58:03,73152,50000,2740998,111383,2629615,2214,42416,0,0


## 4. Negative sampling (train only)

We apply negative sampling to balance the training data given that click rate ratio is only 4% to balance the training data. For each clicked news aarticle, we will sample 4 non-clicked articles from within the same impression to form the negative samples.

For each clicked candidate, draw `K_NEGATIVES` non-clicked candidates from the same impression (`src/sampling.py`). Slot 0 is the click and slots 1..k are negatives, sharing a `group_id`: use rows directly for our model building  (NeuMF, feature MLP) or group by `group_id` for a listwise softmax (candidate-aware attention) model.

Negatives are drawn without replacement when the impression has at least k, otherwise with replacement; clicks in impressions without negatives are dropped.

The exported CSVs stay unsampled. During training, redraw each epoch with `sample_negatives(train_candidates, k=K_NEGATIVES, seed=SAMPLING_SEED + epoch)`.

In [5]:
import pandas as pd
from src.data_loading import load_partition
from src.sampling import sample_negatives

K_NEGATIVES = 4
SAMPLING_SEED = 42

_, train_candidates, _ = load_partition("train", OUTPUT_DIR)
train_sampled = sample_negatives(train_candidates, k=K_NEGATIVES, seed=SAMPLING_SEED)

group_sizes = train_sampled.groupby("group_id").size()
assert group_sizes.eq(K_NEGATIVES + 1).all(), "Every group must hold one click and k negatives"
assert train_sampled.groupby("group_id")["event_key"].nunique().eq(1).all(), "Negatives must share the click's impression"
display(pd.DataFrame({
    "rows": [len(train_candidates), len(train_sampled)],
    "clicks": [int(train_candidates["label"].sum()), int(train_sampled["label"].sum())],
    "click_share": [train_candidates["label"].mean(), train_sampled["label"].mean()],
}, index=["train (full)", "train (sampled)"]))

,rows,clicks,click_share
train (full),3389342,141558,0.041766
train (sampled),707790,141558,0.200000


## 5. Load the same data teammates will use

`load_partition` restores string IDs, timestamps, booleans and JSON lists. Read the exported `data_dictionary.md` before engineering features. Never use labels or current-event click totals as features. Fit learned transforms on training only; compute interaction statistics from permitted past events.

Model training uses `train_sampled` (from section 4) joined to `events` on `event_key` for user history and to `news` on `news_id` for article content. `candidates` stays the full train table for feature statistics.

In [6]:
from src.data_loading import load_partition

events, candidates, news = load_partition("train", OUTPUT_DIR)
display(events.head())
display(candidates.head())
display(news[["news_id", "category", "title", "abstract_missing"]].head())

# Sampled training groups with user and article context.
train_model_input = (
    train_sampled
    .merge(events[["event_key", "user_id", "timestamp", "history_ids"]], on="event_key", how="left", validate="many_to_one")
    .merge(news[["news_id", "category", "subcategory", "title"]], on="news_id", how="left", validate="many_to_one")
)
assert len(train_model_input) == len(train_sampled), "Joins changed the sampled row count"
display(train_model_input.head(2 * (K_NEGATIVES + 1)))

,impression_id,user_id,time,history,timestamp,source_file,event_key,history_ids,history_length,history_missing
0,20112,U65916,11/9/2019 12:00:19 AM,N51706 N40767 N12096 N9798 N38802 N54827 N5780...,2019-11-09 00:00:19,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:20112,"[N51706, N40767, N12096, N9798, N38802, N54827...",15,False
1,13807,U49985,11/9/2019 12:01:13 AM,N5056 N29975 N53234 N39603 N50032 N8422 N53580...,2019-11-09 00:01:13,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:13807,"[N5056, N29975, N53234, N39603, N50032, N8422,...",93,False
2,27660,U25550,11/9/2019 12:02:44 AM,N17260 N38298 N33976 N47719 N14888 N18870 N4607,2019-11-09 00:02:44,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:27660,"[N17260, N38298, N33976, N47719, N14888, N1887...",7,False
3,152217,U19710,11/9/2019 12:02:50 AM,N3530 N48284 N43019 N62546 N138 N13138 N10676 ...,2019-11-09 00:02:50,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:152217,"[N3530, N48284, N43019, N62546, N138, N13138, ...",8,False
4,42166,U38106,11/9/2019 12:03:09 AM,N16874 N264 N48697 N51366,2019-11-09 00:03:09,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:42166,"[N16874, N264, N48697, N51366]",4,False


,event_key,candidate_position,news_id,label
0,MINDsmall_train/behaviors.tsv:20112,0,N54300,0
1,MINDsmall_train/behaviors.tsv:20112,1,N46057,1
2,MINDsmall_train/behaviors.tsv:20112,2,N57005,0
3,MINDsmall_train/behaviors.tsv:20112,3,N52154,0
4,MINDsmall_train/behaviors.tsv:20112,4,N57099,0


,news_id,category,title,abstract_missing
0,N10,health,7 Running Workouts for Weight Loss,False
1,N100,sports,Astros finally get timely hitting to get back ...,False
2,N1000,sports,"Sounders vs. Real Salt Lake: Highlights, stats...",False
3,N10000,sports,"The Day in Sports: Wednesday, October 30, 2019",False
4,N10001,news,Vero Beach man charged with battery in connect...,False


,group_id,event_key,news_id,candidate_position,label,slot,user_id,timestamp,history_ids,category,subcategory,title
0,0,MINDsmall_train/behaviors.tsv:1,N55689,0,1,0,U13740,2019-11-11 09:05:58,"[N55189, N42782, N34694, N45794, N18445, N6330...",sports,football_nfl,"Charles Rogers, former Michigan State football..."
1,0,MINDsmall_train/behaviors.tsv:1,N35729,1,0,1,U13740,2019-11-11 09:05:58,"[N55189, N42782, N34694, N45794, N18445, N6330...",news,newsus,Porsche launches into second story of New Jers...
2,0,MINDsmall_train/behaviors.tsv:1,N35729,1,0,2,U13740,2019-11-11 09:05:58,"[N55189, N42782, N34694, N45794, N18445, N6330...",news,newsus,Porsche launches into second story of New Jers...
3,0,MINDsmall_train/behaviors.tsv:1,N35729,1,0,3,U13740,2019-11-11 09:05:58,"[N55189, N42782, N34694, N45794, N18445, N6330...",news,newsus,Porsche launches into second story of New Jers...
4,0,MINDsmall_train/behaviors.tsv:1,N35729,1,0,4,U13740,2019-11-11 09:05:58,"[N55189, N42782, N34694, N45794, N18445, N6330...",news,newsus,Porsche launches into second story of New Jers...
5,1,MINDsmall_train/behaviors.tsv:10,N53585,0,1,0,U53231,2019-11-11 11:28:11,"[N58936, N15919, N11917, N2153, N55312, N13008...",tv,tvnews,"Rip Taylor's Cause of Death Revealed, Memorial..."
6,1,MINDsmall_train/behaviors.tsv:10,N55689,1,0,1,U53231,2019-11-11 11:28:11,"[N58936, N15919, N11917, N2153, N55312, N13008...",sports,football_nfl,"Charles Rogers, former Michigan State football..."
7,1,MINDsmall_train/behaviors.tsv:10,N55689,1,0,2,U53231,2019-11-11 11:28:11,"[N58936, N15919, N11917, N2153, N55312, N13008...",sports,football_nfl,"Charles Rogers, former Michigan State football..."
8,1,MINDsmall_train/behaviors.tsv:10,N55689,1,0,3,U53231,2019-11-11 11:28:11,"[N58936, N15919, N11917, N2153, N55312, N13008...",sports,football_nfl,"Charles Rogers, former Michigan State football..."
9,1,MINDsmall_train/behaviors.tsv:10,N55689,1,0,4,U53231,2019-11-11 11:28:11,"[N58936, N15919, N11917, N2153, N55312, N13008...",sports,football_nfl,"Charles Rogers, former Michigan State football..."
